In [8]:
import transformers
import datasets
import accelerate
import evaluate
import tqdm

In [9]:
from datasets import load_dataset

# 加载 WMT19 英语-法语数据集
dataset = load_dataset("wmt19", "zh-en")

print(dataset)
#分割
train_val_split=dataset["train"].train_test_split(test_size=0.1,seed=42)
small_train_dataset=train_val_split["test"]
test_dataset=dataset["validation"]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 25984574
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 3981
    })
})


In [3]:
parallel_corpus=[]
for example in small_train_dataset:
    source=example["translation"]['zh']
    target=example["translation"]['en']
    parallel_corpus.append((source,target))

for source,target in parallel_corpus[:5]:
    print(f'source:{source}')
    print(f'target:{target}')

source:加拿大国际开发署为帮助发展中国家实现儿童权利发挥的建设性作用，以及代表团团长宣布到2010年加拿大将把国际援助增加一倍。
target:The constructive role played by the Canadian International Development Agency (CIDA) to assist developing countries in fulfilling the rights of their children and the declaration by the head of the delegation that Canada will double its international aid by 2010.
source:3. 2002年3月1日至5日，特别报告员访问喀土穆，他在喀土穆会见了下列人士：第一副总统、和平问题总统顾问、司法、国防、能源和矿业、新闻和传播、咨询和捐赠事务部长、外交国务部长和国民议会人权委员会主席、以及人权咨询理事会报告员。
target:3. From 1 through 5 March 2002, the Special Rapporteur visited Khartoum, where he met with the First Vice-President, the Presidential Adviser on peace, the Ministers of Justice, Defence, Energy and Mining, Information and Communication, Guidance and Endowment, the State Minister for Foreign Affairs and the Head of the Human Rights Committee in the National Assembly, as well as the Rapporteur of the Advisory Council for Human Rights.
source:难民和国内流离失所者回返
target:B. Return of refugees and internally displaced persons
source:要想做到这

In [4]:
with open('parallel_corpus.txt','w',encoding='utf-8') as f:
    for source,target in parallel_corpus:
        f.write(f'{source}\t{target}\n')

In [5]:
import csv
with open('parallel_corpus.csv','w',encoding='utf-8') as f:
    writer=csv.writer(f)
    writer.writerow(['source','target'])
    for source,target in parallel_corpus:
        writer.writerow([source,target])

SentencePiece方法

In [ ]:
import sentencepiece as spm
spm.SentencePieceTrainer.train(
    input='parallel_corpus.txt',
    model_prefix='tokenizer_model',
    vocab_size=50000,
    character_coverage=1.0,
    model_type='bpe'
)

In [7]:
spm_model=spm.SentencePieceProcessor(model_file='tokenizer_model.model')
encoded=spm_model.EncodeAsIds('我爱你')
encoded_1=spm_model.EncodeAsIds('I love you')
print(encoded)
decoded=spm_model.DecodeIds(encoded)
print(decoded)

[597, 24371]
我爱你


Hugging Face tokenizers方法

In [11]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer=Tokenizer(BPE())
tokenizer.pre_tokenizer=Whitespace()

trainer=BpeTrainer(
    vocab_size=50000,
    min_frequency=2,
    show_progress=True
)
tokenizer.train(files=["parallel_corpus.txt"],trainer=trainer)
tokenizer.save("hf_bpe_tokenizer.json")

In [15]:
tokenizer=Tokenizer.from_file("hf_bpe_tokenizer.json")
encoded=tokenizer.encode('hello, how are you')
print(encoded.tokens)
decoded=tokenizer.decode(encoded.ids)
print(decoded)

['hel', 'lo', ',', 'how', 'are', 'you']
hel lo , how are you


In [12]:
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer

WP_tokenizer=Tokenizer(WordPiece())
WP_tokenizer.pre_tokenizer=Whitespace()
WP_trainer=WordPieceTrainer(
    vocab_size=50000,
    min_frequency=2,
    show_progress=True
)
WP_tokenizer.train(files=["parallel_corpus.txt"],trainer=WP_trainer)
WP_tokenizer.save("hf_wp_tokenizer.json")

In [16]:
tokenizer=Tokenizer.from_file("hf_wp_tokenizer.json")
encoded=tokenizer.encode('hello, how are you')
print(encoded.tokens)
decoded=tokenizer.decode(encoded.ids)
print(decoded)

['hell', '##o', ',', 'how', 'are', 'you']
hell ##o , how are you


In [19]:
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer

Unigram_tokenizer=Tokenizer(Unigram())
Unigram_tokenizer.pre_tokenizer=Whitespace()
Unigram_trainer=UnigramTrainer(
    vocab_size=50000,
    min_frequency=2,
    show_progress=True
)
Unigram_tokenizer.train(files=["parallel_corpus.txt"],trainer=Unigram_trainer)
Unigram_tokenizer.save("hf_unigram_tokenizer.json")

In [21]:
tokenizer=Tokenizer.from_file("hf_unigram_tokenizer.json")
encoded=tokenizer.encode('hello, how are you')
print(encoded.ids)
decoded=tokenizer.decode(encoded.ids)
print(decoded)

[12788, 31, 3, 563, 46, 99]
hell o , how are you


In [8]:
class CustomTokenizer:
    def __init__(self,model_type='bpe'):
        self.model_type=model_type.lower()

        if self.model_type == 'bpe':
            self.tokenizer=Tokenizer(BPE())
            self.trainer=BpeTrainer(vocab_size=50000,min_frequency=2,show_progress=True)
        elif self.model_type == 'wordpiece':
            self.tokenizer=Tokenizer(WordPiece())
            self.trainer=WordPieceTrainer(vocab_size=50000,min_frequency=2,show_progress=True)
        elif self.model_type == 'unigram':
            self.tokenizer=Tokenizer(Unigram())
            self.trainer=Unigram_trainer(vocab_size=50000,min_frequency=2,show_progress=True)
        else:
            raise ValueError(f"Unsupport model type: {self.model_type}")
        
        self.tokenizer.pre_tokenizer=Whitespace()

    def train(self,files):
        self.tokenizer.train(files,self.trainer)
    
    def save(self,model_path):
        self.tokenizer.save(model_path)
    
    def encode(self,text):
        return self.tokenizer.encode(text)
    
    def decode(self,token_ids):
        return self.tokenizer.decode(token_ids)
    
    def load(self,model_path):
        self.tokenizer=Tokenizer.from_file(model_path)